# Step 2: Data Cleaning, Validation & Textual Return Reason Profiling
**MSc Data Science Thesis - University of Wolverhampton**

###  Objective & Methodological Processing
Perform data validation, drop incomplete pricing/ID observations, parse datetime columns, and extract operational return reasons.

###  Features Implemented:
1. **Observation Filtering**: Keeps validated order statuses (`delivered` and `canceled`) and drops missing pricing keys.
2. **Categorical Textual Return Reason Parser (`return_reason`)**:
   - `Product Quality/Damage` (defects, faults, broken, cracked, torn)
   - `Logistics/Delivery Issue` (late, delayed, slow delivery, never arrived)
   - `Fulfillment Error` (wrong item, mistake, size, color)
   - `Price/Value Issue` (expensive, overpriced, not worth, refund)
   - `Other/General Dissatisfaction` / `No Review`
3. **Package Volumetrics & Delay Metrics**: Calculates package volume ($L \times W \times H$), shipping ratio, and delivery delay days.
4. **Output Checkpoint**: `consolidated_cleaned_data.csv`.

In [1]:
import pandas as pd
import numpy as np

print("Step 2: Commencing data validation and processing pipeline...")
df = pd.read_csv('consolidated_return_data.csv', encoding='latin-1')

# Drop observations missing core tracking attributes or pricing keys
df = df.dropna(subset=['price', 'freight_value', 'order_id', 'customer_unique_id', 'is_returned'])
df = df[df['order_status'].isin(['delivered', 'canceled'])]

# Parse raw dates cleanly to datetime format
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Engineering basic logistics parameters
df['product_volume_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['delivery_delay_days'] = df['delivery_delay_days'].fillna(0)
df['shipping_cost_ratio'] = df['freight_value'] / df['price']
df['is_shipping_more_than_item'] = (df['freight_value'] > df['price']).astype(int)

print("Extracting categorized textual return justification flags...")
def parse_return_justification(text):
    if pd.isna(text) or text == 'No comment': return "No Review"
    text = str(text).lower()
    if any(w in text for w in ['broken', 'damage','damaged', 'defect','defective', 'fault','faulty','doesnt work', 'not work','scratched', 'not working', 'cracked', 'torn']): return 'Product Quality/Damage'
    if any(w in text for w in ['late', 'delayed', 'delivery', 'wait', 'never arrived', 'slow']): return 'Logistics/Delivery Issue'
    if any(w in text for w in ['wrong', 'different', 'mistake', 'size', 'color']): return 'Fulfillment Error'
    if any(w in text for w in ['expensive', 'overpriced', 'not worth', 'refund']): return 'Price/Value Issue'
    return 'Other/General Dissatisfaction'

df['return_reason'] = df['review_comment_message_english'].apply(parse_return_justification)

# Impute dimensional variables with category medians
dimension_columns = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_volume_cm3']
for col in dimension_columns:
    df[col] = df.groupby('product_category_name_english')[col].transform(lambda x: x.fillna(x.median())).fillna(df[col].median())

df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

# Save step 2 data to explicit file checkpoint for Step 3 to read
df.to_csv('consolidated_cleaned_data.csv', index=False)
print("Success! Step 2 clean checkpoint saved to 'consolidated_cleaned_data.csv'.\n")

Step 2: Commencing data validation and processing pipeline...
Extracting categorized textual return justification flags...
Success! Step 2 clean checkpoint saved to 'consolidated_cleaned_data.csv'.

